# Notebook 04 -- Final Results, Statistical Tests & Publication Figures

**Research:** Hybrid AO-PSO Optimised Temporal Fusion Transformer for Smart Grid Multi-Horizon Energy Forecasting

---

## Purpose

This notebook performs **no training**. It aggregates the pre-computed 10x Grind results from Notebooks 01--03 and produces:

1. **Summary Table** -- Mean ± Std of RMSE, MAE, MAPE for all 6 models across both datasets.
2. **Wilcoxon Signed-Rank Tests** -- Pairwise statistical significance of the Champion (Hybrid AO-PSO-TFT) vs. every baseline and standalone optimizer.
3. **Figure 1: Robustness Boxplots** -- Distribution of test RMSE across 10 independent runs.
4. **Figure 2: Convergence Curves** -- Median proxy fitness over optimizer iterations for AO, PSO, and Hybrid (with hand-off annotation).
5. **Figure 3: Horizon Degradation** -- RMSE at 1h, 6h, 12h, and 24h forecast horizons for all models.

All figures are rendered at 300 DPI with academic styling for direct inclusion in the Q1 manuscript.

### Input Files

| File | Source Notebook | Contents |
|------|----------------|----------|
| `results/baseline_metrics.json` | NB 01 | LSTM, BiLSTM, XGBoost × 10 runs × 2 datasets |
| `results/standalone_swarm_metrics.json` | NB 02 | Pure AO, Pure PSO × 10 runs × 2 datasets + convergence |
| `results/hybrid_metrics.json` | NB 03 | Hybrid AO-PSO × 10 runs × 2 datasets + convergence (with phase labels) |

In [ ]:
# ================================================================
# Cell 1: Imports & Style
# ================================================================
import sys
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# ---- Set project root ----
PROJECT_ROOT = "/workspace/Hybrid-Optimizer-AO-PSO"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from src.metrics import run_wilcoxon_test

# ---- Academic plot style ----
sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.dpi":        150,
    "savefig.dpi":       300,
    "font.size":         11,
    "axes.titlesize":    13,
    "axes.labelsize":    12,
    "legend.fontsize":   10,
    "xtick.labelsize":   10,
    "ytick.labelsize":   10,
    "font.family":       "serif",
    "mathtext.fontset":  "cm",
})

RESULTS_DIR = Path("results")
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

# ---- Canonical model order and colour palette ----
MODEL_ORDER = [
    "StandardLSTM", "BiLSTM", "XGBoost",
    "AquilaOptimizer", "ParticleSwarm", "Hybrid_AO_PSO",
]
MODEL_LABELS = {
    "StandardLSTM":    "LSTM",
    "BiLSTM":          "Bi-LSTM",
    "XGBoost":         "XGBoost",
    "AquilaOptimizer": "AO-TFT",
    "ParticleSwarm":   "PSO-TFT",
    "Hybrid_AO_PSO":   "Hybrid AO-PSO-TFT",
}
PALETTE = {
    "LSTM":               "#64B5F6",
    "Bi-LSTM":            "#42A5F5",
    "XGBoost":            "#66BB6A",
    "AO-TFT":             "#FFA726",
    "PSO-TFT":            "#EF5350",
    "Hybrid AO-PSO-TFT":  "#AB47BC",
}

print(f"Results dir : {RESULTS_DIR.resolve()}")
print(f"Figures dir : {FIGURES_DIR.resolve()}")

In [ ]:
# ================================================================
# Cell 2: Data Loading
# ================================================================

def load_results():
    """
    Load the three JSON result files and parse them into a unified
    DataFrame with columns: Dataset, Model, Run, RMSE, MAE, MAPE, Time.

    Also returns the raw dicts for convergence/horizon extraction.
    Handles missing files gracefully.
    """
    raw = {}
    files = {
        "baseline":  RESULTS_DIR / "baseline_metrics.json",
        "swarm":     RESULTS_DIR / "standalone_swarm_metrics.json",
        "hybrid":    RESULTS_DIR / "hybrid_metrics.json",
    }

    for key, path in files.items():
        if path.exists():
            with open(path) as f:
                raw[key] = json.load(f)
            print(f"  Loaded {path.name} ({path.stat().st_size / 1024:.1f} KB)")
        else:
            print(f"  WARNING: {path.name} not found -- skipping.")
            raw[key] = {}

    # Parse into rows
    rows = []
    for dataset in ["micro", "macro"]:
        for source_key, source_data in raw.items():
            if dataset not in source_data:
                continue
            for model_name, run_list in source_data[dataset].items():
                if not isinstance(run_list, list):
                    continue
                for r in run_list:
                    # baseline uses "time_s", swarm/hybrid use "total_time_s"
                    t = r.get("total_time_s", r.get("time_s", 0))
                    rows.append({
                        "Dataset": dataset.capitalize(),
                        "Model":   model_name,
                        "Run":     r["run"],
                        "RMSE":    r["RMSE"],
                        "MAE":     r["MAE"],
                        "MAPE":    r["MAPE"],
                        "Time":    t,
                    })

    df = pd.DataFrame(rows)

    # Add display labels
    df["Model_Label"] = df["Model"].map(MODEL_LABELS)

    return df, raw


df, raw_data = load_results()

print(f"\nTotal rows: {len(df)}")
print(f"Datasets:   {df['Dataset'].unique().tolist()}")
print(f"Models:     {df['Model'].unique().tolist()}")

# ---- Summary table: Mean +/- Std ----
summary = (
    df.groupby(["Dataset", "Model"])
    .agg(
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        MAPE_mean=("MAPE", "mean"),
        MAPE_std=("MAPE", "std"),
        Time_mean=("Time", "mean"),
        N=("Run", "count"),
    )
    .reset_index()
)

# Format for display
summary["RMSE"]  = summary.apply(lambda r: f"{r.RMSE_mean:.4f} \u00b1 {r.RMSE_std:.4f}", axis=1)
summary["MAE"]   = summary.apply(lambda r: f"{r.MAE_mean:.4f} \u00b1 {r.MAE_std:.4f}", axis=1)
summary["MAPE"]  = summary.apply(lambda r: f"{r.MAPE_mean:.2f}% \u00b1 {r.MAPE_std:.2f}%", axis=1)
summary["Label"] = summary["Model"].map(MODEL_LABELS)

display_cols = ["Dataset", "Label", "RMSE", "MAE", "MAPE", "N"]
print("\n" + "=" * 80)
print("  SUMMARY TABLE: Mean \u00b1 Std (10x Grind Protocol)")
print("=" * 80)
print(summary[display_cols].to_string(index=False))

In [ ]:
# ================================================================
# Cell 3: Statistical Testing (Wilcoxon Signed-Rank)
# ================================================================
# For each dataset, compare the Hybrid AO-PSO's 10 RMSE values
# against every other model using a one-sided test
# (alternative="greater" -> H1: baseline RMSE > hybrid RMSE,
#  i.e., the hybrid is better).

COMPARISONS = [
    "StandardLSTM", "BiLSTM", "XGBoost",
    "AquilaOptimizer", "ParticleSwarm",
]

wilcoxon_rows = []

for dataset in ["Micro", "Macro"]:
    hybrid_rmses = df.loc[
        (df["Dataset"] == dataset) & (df["Model"] == "Hybrid_AO_PSO"), "RMSE"
    ].values

    if len(hybrid_rmses) < 5:
        print(f"  WARNING: {dataset} -- Hybrid has {len(hybrid_rmses)} runs "
              f"(need >= 5 for Wilcoxon). Skipping.")
        continue

    for comp_model in COMPARISONS:
        comp_rmses = df.loc[
            (df["Dataset"] == dataset) & (df["Model"] == comp_model), "RMSE"
        ].values

        if len(comp_rmses) < 5:
            continue

        # Test: is the baseline's RMSE greater than the hybrid's?
        result = run_wilcoxon_test(
            comp_rmses, hybrid_rmses, alternative="greater",
        )

        wilcoxon_rows.append({
            "Dataset":     dataset,
            "Comparison":  f"{MODEL_LABELS[comp_model]} vs Hybrid",
            "W-Statistic": result["W_statistic"],
            "p-value":     result["p_value"],
            "Significant":  "Yes" if result["p_value"] < 0.05 else "No",
        })

if wilcoxon_rows:
    wilcoxon_df = pd.DataFrame(wilcoxon_rows)
    wilcoxon_df["p-value"] = wilcoxon_df["p-value"].map(
        lambda p: f"{p:.6f}" if p >= 0.001 else f"{p:.2e}"
    )
    print("\n" + "=" * 80)
    print("  WILCOXON SIGNED-RANK TEST (H1: Baseline RMSE > Hybrid RMSE)")
    print("  Significance level: \u03b1 = 0.05")
    print("=" * 80)
    print(wilcoxon_df.to_string(index=False))
else:
    print("  No Wilcoxon tests could be run (insufficient data).")

In [ ]:
# ================================================================
# Cell 4: Figure 1 -- Robustness Boxplots
# ================================================================
# Side-by-side boxplots showing RMSE distribution across 10 runs
# for all models. One subplot per dataset.

datasets_available = [d for d in ["Micro", "Macro"]
                      if d in df["Dataset"].unique()]
n_plots = len(datasets_available)

if n_plots == 0:
    print("No data available for boxplots.")
else:
    fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 5), squeeze=False)

    for i, dataset in enumerate(datasets_available):
        ax = axes[0, i]
        subset = df[df["Dataset"] == dataset].copy()

        # Order models canonically (only those present)
        present = [m for m in MODEL_ORDER if m in subset["Model"].unique()]
        labels  = [MODEL_LABELS[m] for m in present]
        subset["Model_Label"] = pd.Categorical(
            subset["Model"].map(MODEL_LABELS), categories=labels, ordered=True,
        )

        sns.boxplot(
            data=subset, x="Model_Label", y="RMSE",
            palette=PALETTE, ax=ax, width=0.5,
            flierprops={"marker": "o", "markersize": 4, "alpha": 0.6},
        )

        dataset_label = ("Micro-Grid (UCI Household)" if dataset == "Micro"
                         else "Macro-Grid (ISO-NE)")
        ax.set_title(dataset_label, fontweight="bold")
        ax.set_xlabel("")
        ax.set_ylabel("Test RMSE (real-world units)")
        ax.tick_params(axis="x", rotation=25)

    fig.suptitle("Figure 1: Forecasting Robustness -- RMSE Distribution (10 Runs)",
                 fontsize=14, fontweight="bold", y=1.02)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "fig1_robustness_boxplots.png",
                dpi=300, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {FIGURES_DIR / 'fig1_robustness_boxplots.png'}")

In [ ]:
# ================================================================
# Cell 5: Figure 2 -- Convergence Curves
# ================================================================
# Extract convergence histories from the swarm and hybrid JSONs.
# For each optimizer, compute the median best_fitness at each
# iteration across the 10 runs. Plot with the AO->PSO hand-off.

OPTIMIZER_STYLES = {
    "AquilaOptimizer": {"color": "#FFA726", "marker": "^", "label": "Pure AO"},
    "ParticleSwarm":   {"color": "#EF5350", "marker": "s", "label": "Pure PSO"},
    "Hybrid_AO_PSO":   {"color": "#AB47BC", "marker": "D", "label": "Hybrid AO-PSO"},
}


def extract_convergence(raw_data, dataset_key):
    """
    Extract convergence curves from swarm and hybrid raw data.

    Returns: {opt_name: np.ndarray of shape (n_runs, n_iters)} for best_fitness.
    """
    curves = {}

    for source_key in ["swarm", "hybrid"]:
        source = raw_data.get(source_key, {})
        if dataset_key not in source:
            continue
        for opt_name, run_list in source[dataset_key].items():
            if not isinstance(run_list, list):
                continue

            run_curves = []
            for r in run_list:
                if "convergence" not in r:
                    continue
                history = r["convergence"].get("history", [])
                if history:
                    run_curves.append([h["best_fitness"] for h in history])

            if run_curves:
                # Pad to equal length (in case some runs converged early)
                max_len = max(len(c) for c in run_curves)
                padded = []
                for c in run_curves:
                    if len(c) < max_len:
                        c = c + [c[-1]] * (max_len - len(c))
                    padded.append(c)
                curves[opt_name] = np.array(padded)

    return curves


def get_handoff_iter(raw_data, dataset_key):
    """Get the hand-off iteration from the first Hybrid run's convergence."""
    hybrid = raw_data.get("hybrid", {})
    if dataset_key not in hybrid or "Hybrid_AO_PSO" not in hybrid[dataset_key]:
        return None
    runs = hybrid[dataset_key]["Hybrid_AO_PSO"]
    if not runs or "convergence" not in runs[0]:
        return None
    history = runs[0]["convergence"].get("history", [])
    ao_iters = [h["iteration"] for h in history if h.get("phase") == "AO"]
    return max(ao_iters) if ao_iters else None


# ---- Plot ----
datasets_available = [d for d in ["micro", "macro"]
                      if any(d in raw_data.get(k, {}) for k in ["swarm", "hybrid"])]

n_plots = len(datasets_available)

if n_plots == 0:
    print("No convergence data available.")
else:
    fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 5), squeeze=False)

    for i, dataset_key in enumerate(datasets_available):
        ax = axes[0, i]
        curves = extract_convergence(raw_data, dataset_key)
        handoff = get_handoff_iter(raw_data, dataset_key)

        for opt_name in ["AquilaOptimizer", "ParticleSwarm", "Hybrid_AO_PSO"]:
            if opt_name not in curves:
                continue
            arr = curves[opt_name]
            median_curve = np.median(arr, axis=0)
            q25 = np.percentile(arr, 25, axis=0)
            q75 = np.percentile(arr, 75, axis=0)
            iters = np.arange(1, len(median_curve) + 1)

            style = OPTIMIZER_STYLES[opt_name]
            ax.plot(iters, median_curve, marker=style["marker"],
                    color=style["color"], linewidth=2, markersize=4,
                    label=style["label"], zorder=3)
            ax.fill_between(iters, q25, q75, color=style["color"],
                            alpha=0.15, zorder=1)

        # Hand-off line
        if handoff is not None:
            ax.axvline(x=handoff + 0.5, color="#333333", linestyle="--",
                       linewidth=1.5, alpha=0.7, label="AO\u2192PSO Hand-Off")
            ax.annotate("HAND-OFF", xy=(handoff + 0.5, ax.get_ylim()[1]),
                        xytext=(handoff + 1, ax.get_ylim()[1] * 0.97),
                        fontsize=8, fontweight="bold", color="#333333",
                        ha="left", va="top")

        dataset_label = ("Micro-Grid (UCI Household)" if dataset_key == "micro"
                         else "Macro-Grid (ISO-NE)")
        ax.set_title(dataset_label, fontweight="bold")
        ax.set_xlabel("Optimizer Iteration")
        ax.set_ylabel("Best Proxy Validation RMSE (scaled)")
        ax.legend(loc="upper right", framealpha=0.9)
        ax.grid(True, alpha=0.3)

    fig.suptitle("Figure 2: Optimizer Convergence -- Median over 10 Runs (IQR shaded)",
                 fontsize=14, fontweight="bold", y=1.02)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "fig2_convergence_curves.png",
                dpi=300, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {FIGURES_DIR / 'fig2_convergence_curves.png'}")

In [ ]:
# ================================================================
# Cell 6: Figure 3 -- Horizon Degradation
# ================================================================
# Extract per-horizon RMSE (1h, 6h, 12h, 24h) for all models.
# Plot the mean RMSE at each horizon as a line chart.

HORIZONS = [1, 6, 12, 24]


def extract_horizon_data(raw_data, dataset_key):
    """
    Extract per-horizon RMSE averages from all three JSON sources.

    Returns: {model_name: {horizon: mean_rmse}}
    """
    horizon_data = {}

    for source_key in ["baseline", "swarm", "hybrid"]:
        source = raw_data.get(source_key, {})
        if dataset_key not in source:
            continue
        for model_name, run_list in source[dataset_key].items():
            if not isinstance(run_list, list):
                continue

            # Collect per-run per-horizon RMSE
            per_horizon = {h: [] for h in HORIZONS}
            for r in run_list:
                hm = r.get("horizon_metrics", {})
                for h in HORIZONS:
                    h_data = hm.get(str(h), {})
                    if "RMSE" in h_data:
                        per_horizon[h].append(h_data["RMSE"])

            # Average across runs
            if any(per_horizon[h] for h in HORIZONS):
                horizon_data[model_name] = {
                    h: np.mean(per_horizon[h]) if per_horizon[h] else np.nan
                    for h in HORIZONS
                }

    return horizon_data


# ---- Plot ----
datasets_available = [d for d in ["micro", "macro"]
                      if any(d in raw_data.get(k, {}) for k in ["baseline", "swarm", "hybrid"])]
n_plots = len(datasets_available)

if n_plots == 0:
    print("No horizon data available.")
else:
    fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 5), squeeze=False)

    MARKER_MAP = {
        "StandardLSTM":    "o",
        "BiLSTM":          "v",
        "XGBoost":         "s",
        "AquilaOptimizer": "^",
        "ParticleSwarm":   "D",
        "Hybrid_AO_PSO":   "*",
    }

    for i, dataset_key in enumerate(datasets_available):
        ax = axes[0, i]
        horizon_data = extract_horizon_data(raw_data, dataset_key)

        for model_name in MODEL_ORDER:
            if model_name not in horizon_data:
                continue
            hd = horizon_data[model_name]
            rmses = [hd[h] for h in HORIZONS]
            label = MODEL_LABELS[model_name]

            linewidth = 2.5 if model_name == "Hybrid_AO_PSO" else 1.5
            markersize = 9 if model_name == "Hybrid_AO_PSO" else 6

            ax.plot(HORIZONS, rmses, marker=MARKER_MAP.get(model_name, "o"),
                    color=PALETTE[label], linewidth=linewidth,
                    markersize=markersize, label=label, zorder=3)

        dataset_label = ("Micro-Grid (UCI Household)" if dataset_key == "micro"
                         else "Macro-Grid (ISO-NE)")
        ax.set_title(dataset_label, fontweight="bold")
        ax.set_xlabel("Forecast Horizon (hours)")
        ax.set_ylabel("RMSE (real-world units)")
        ax.set_xticks(HORIZONS)
        ax.set_xticklabels([f"{h}h" for h in HORIZONS])
        ax.legend(loc="upper left", framealpha=0.9)
        ax.grid(True, alpha=0.3)

    fig.suptitle("Figure 3: Forecast Accuracy Degradation Across Horizons (Mean of 10 Runs)",
                 fontsize=14, fontweight="bold", y=1.02)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "fig3_horizon_degradation.png",
                dpi=300, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {FIGURES_DIR / 'fig3_horizon_degradation.png'}")

In [ ]:
# ================================================================
# Cell 7: Export Summary Tables to CSV
# ================================================================
# Save the summary and Wilcoxon tables as CSV for easy inclusion
# in the LaTeX manuscript via \input or pandas-to-LaTeX.

# --- Summary table ---
summary_export = (
    df.groupby(["Dataset", "Model"])
    .agg(
        RMSE_mean=("RMSE", "mean"), RMSE_std=("RMSE", "std"),
        MAE_mean=("MAE", "mean"),   MAE_std=("MAE", "std"),
        MAPE_mean=("MAPE", "mean"), MAPE_std=("MAPE", "std"),
        Time_mean=("Time", "mean"), N=("Run", "count"),
    )
    .reset_index()
)
summary_export["Label"] = summary_export["Model"].map(MODEL_LABELS)
summary_path = RESULTS_DIR / "summary_table.csv"
summary_export.to_csv(summary_path, index=False)
print(f"Summary table saved to: {summary_path}")

# --- Wilcoxon table ---
if wilcoxon_rows:
    wilcoxon_path = RESULTS_DIR / "wilcoxon_tests.csv"
    pd.DataFrame(wilcoxon_rows).to_csv(wilcoxon_path, index=False)
    print(f"Wilcoxon table saved to: {wilcoxon_path}")

print("\n" + "=" * 60)
print("  ALL ANALYSIS COMPLETE")
print("=" * 60)
print(f"  Figures : {list(FIGURES_DIR.glob('*.png'))}")
print(f"  Tables  : {list(RESULTS_DIR.glob('*.csv'))}")
print("\n>>> NOTEBOOK 04 COMPLETE <<<")